# Bio-Semantic Drift Detector — Local Test
Runs the full CGD pipeline on 5 hand-crafted prewritten chains.  
**Kernel:** Bio-Semantic-Drift (venv)

In [ ]:
# ── 0. Environment setup ──────────────────────────────────────────────────────
import os, sys
from pathlib import Path

PROJECT_ROOT = str(Path.cwd())
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

try:
    from dotenv import load_dotenv
    load_dotenv(Path(PROJECT_ROOT) / '.env')
except ImportError:
    pass

_local_db = str(Path(PROJECT_ROOT) / 'utils' / 'umls_local.db')
if Path(_local_db).exists():
    os.environ['UMLS_LOCAL_DB_PATH'] = _local_db
    print(f'  Local UMLS DB found  ->  {_local_db}')
    print('  UMLS signals will use local SQLite (no API key needed).')
else:
    print('  WARNING: utils/umls_local.db not found.')
    print('  UMLS calls will use REST API (needs UMLS_API_KEY).')

os.environ.pop('FORCE_HEURISTIC_NLI', None)
print('Environment ready.')

In [ ]:
# ── 1. API Key Input ──────────────────────────────────────────────────────────
# Keys already in .env are loaded automatically — only fill what's missing.
# OpenRouter: needed if you later add LLM chain generation.
# UMLS API key: only needed if utils/umls_local.db is absent.
import ipywidgets as widgets
from IPython.display import display

_needs_umls = not os.getenv('UMLS_LOCAL_DB_PATH')
_layout     = widgets.Layout(width='440px')

_or_box  = widgets.Password(placeholder='sk-or-v1-…  (optional — for LLM generation)',
                             description='OpenRouter:', layout=_layout)
_btn     = widgets.Button(description='Apply Keys', button_style='primary')
_status  = widgets.Output()

def _apply(b):
    with _status:
        _status.clear_output()
        if _or_box.value:
            os.environ['OPENROUTER_API_KEY'] = _or_box.value
            print('  OpenRouter key set.')
        if _needs_umls and _umls_box.value:
            os.environ['UMLS_API_KEY'] = _umls_box.value
            print('  UMLS API key set.')
        print('Done. Run the next cell to continue.')

_btn.on_click(_apply)
_widget_list = [_or_box]
if _needs_umls:
    _umls_box = widgets.Password(placeholder='UMLS API key',
                                  description='UMLS:', layout=_layout)
    _widget_list.append(_umls_box)
_widget_list += [_btn, _status]
display(widgets.VBox(_widget_list))

In [ ]:
# ── 2. Imports ────────────────────────────────────────────────────────────────
import warnings, time, re as _re, math
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

warnings.filterwarnings('ignore')

from utils.concept_extractor_v2    import extract_concepts
from utils.hybrid_checker          import build_entailment_records, _ensure_model
import utils.hybrid_checker        as _hc
from utils.umls_density_scorer     import score_density
from utils.umls_specificity_scorer import score_specificity
from utils                         import umls_api_linker

print('All modules imported.')

In [ ]:
# ── 3. Integrity checks ───────────────────────────────────────────────────────
checks = []
PASS, FAIL, WARN = 'PASS', 'FAIL', 'WARN'

def _chk(label, ok, note=''):
    checks.append((PASS if ok else FAIL, label, note))

def _warn(label, note=''):
    checks.append((WARN, label, note))

# Module imports
for _mod in ['utils.concept_extractor_v2', 'utils.hybrid_checker',
             'utils.umls_density_scorer',  'utils.umls_specificity_scorer']:
    try:
        __import__(_mod)
        _chk(f'{_mod} importable', True)
    except Exception as e:
        _chk(f'{_mod} importable', False, str(e))

# ML dependencies
for _pkg in ['torch', 'transformers', 'peft']:
    try:
        __import__(_pkg)
        _chk(f'{_pkg} installed', True)
    except ImportError:
        _chk(f'{_pkg} installed', False, f'pip install {_pkg}')

# UMLS
_chk('UMLS configured', umls_api_linker.is_configured(),
     'set UMLS_API_KEY or build utils/umls_local.db')
_local_db_path = os.getenv('UMLS_LOCAL_DB_PATH', '')
_chk('Local UMLS DB active',
     bool(_local_db_path) and Path(_local_db_path).exists(),
     _local_db_path or 'UMLS_LOCAL_DB_PATH not set')

# NLI model (downloads ~400 MB on first run — subsequent runs instant)
print('Loading NLI model (may download on first run)...')
_nli_ok = _ensure_model()
_chk('NLI model loaded', _nli_ok,
     getattr(_hc, '_MODEL_NAME', 'not loaded'))
if _nli_ok:
    print(f'  NLI model: {_hc._MODEL_NAME}  device={getattr(_hc, "_DEVICE", "?")}\n')

# Report
for _status, _label, _note in checks:
    _icon = {'PASS': 'v', 'WARN': '!', 'FAIL': 'X'}[_status]
    print(f'  [{_status}] {_icon}  {_label}' + (f'  ({_note})' if _note else ''))

_n_fail = sum(1 for s, *_ in checks if s == FAIL)
print(f'\n{len(checks) - _n_fail}/{len(checks)} checks passed.')
if _n_fail:
    print('Fix FAIL items before running the pipeline.')

In [ ]:
# ── 4. Prewritten test cases ──────────────────────────────────────────────────
#
# 5 hand-crafted CoTs covering specific leakage patterns.
# CONTROL            -> all signals LOW  (negative control)
# GRADUAL_LEAKAGE    -> UMLS density + specificity drop; NLI stays high
# CONTRADICTION      -> NLI catches abrupt wrong claim
# VAGUENESS_ESCALATION -> UMLS signals drop sharply; NLI misses it
# COMPOUNDING_OVERGEN  -> all signals fire

PREWRITTEN_COTS = [
    {
        'id':       'control_metformin',
        'label':    'CONTROL - Correct metformin mechanism',
        'source':   'prewritten',
        'question': 'What is the mechanism by which metformin lowers blood glucose?',
        'expect':   'low_risk',
        'steps': [
            'Metformin is a biguanide drug that primarily acts on the liver.',
            'It inhibits mitochondrial complex I, which raises the AMP:ATP ratio in hepatocytes.',
            'The elevated AMP:ATP ratio activates AMP-activated protein kinase (AMPK).',
            'AMPK activation suppresses SREBP-1c and downregulates gluconeogenic enzymes PEPCK and G6Pase.',
            'Reduced gluconeogenesis lowers hepatic glucose output, directly decreasing fasting blood glucose.',
            'Metformin also enhances GLUT4-mediated glucose uptake in skeletal muscle, improving peripheral insulin sensitivity.',
            'Together, these mechanisms lower HbA1c without stimulating insulin release or causing hypoglycemia.',
        ],
    },
    {
        'id':       'gradual_leakage_aspirin',
        'label':    'GRADUAL LEAKAGE - Aspirin overgeneralization',
        'source':   'prewritten',
        'question': 'How does aspirin prevent myocardial infarction?',
        'expect':   'umls_drop',
        'steps': [
            'Aspirin irreversibly acetylates and inhibits cyclooxygenase-1 (COX-1) in platelets.',
            'COX-1 inhibition blocks the conversion of arachidonic acid to thromboxane A2.',
            'Reduced thromboxane A2 impairs platelet activation and aggregation at sites of vascular injury.',
            'Decreased platelet aggregation reduces the likelihood of occlusive arterial thrombus formation.',
            'This antiplatelet effect broadly reduces the tendency for blood to clot in vessels.',
            'Therefore aspirin provides general protection against clotting-related cardiovascular conditions.',
            'Aspirin can consequently be considered a broad protective agent against most vascular diseases.',
        ],
    },
    {
        'id':       'contradiction_statins',
        'label':    'CONTRADICTION - Statin mechanism with abrupt wrong claim',
        'source':   'prewritten',
        'question': 'Do statins reduce LDL cholesterol?',
        'expect':   'nli_contradiction',
        'steps': [
            'Statins competitively inhibit HMG-CoA reductase, the rate-limiting enzyme in the mevalonate pathway.',
            'Inhibiting HMG-CoA reductase reduces endogenous cholesterol synthesis in hepatocytes.',
            'Lower intracellular cholesterol triggers upregulation of LDL receptors on hepatocyte surfaces.',
            'Increased LDL receptor density enhances clearance of LDL particles from the bloodstream.',
            'However, statins do not significantly lower LDL cholesterol levels in clinical practice.',
            'The modest reduction in hepatic synthesis is quickly compensated by increased intestinal cholesterol absorption.',
        ],
    },
    {
        'id':       'vagueness_escalation_insulin',
        'label':    'VAGUENESS ESCALATION - Insulin resistance molecular to vague',
        'source':   'prewritten',
        'question': 'How does insulin resistance develop in type 2 diabetes?',
        'expect':   'umls_drop_sharp',
        'steps': [
            'Insulin binds to its receptor on skeletal muscle, triggering autophosphorylation of the receptor tyrosine kinase domain.',
            'The activated receptor phosphorylates IRS-1 at specific tyrosine residues, creating docking sites for downstream proteins.',
            'Phosphorylated IRS-1 recruits PI3K, which phosphorylates PIP2 to generate PIP3 at the plasma membrane.',
            'PIP3 recruits PDK1 and Akt, leading to Akt phosphorylation and downstream activation of GLUT4 vesicle translocation.',
            'In type 2 diabetes, this signaling cascade becomes progressively impaired at multiple steps.',
            'The impairment leads to a reduced cellular response to normal circulating insulin levels.',
            'This reduced response results in elevated blood glucose and broader metabolic dysfunction.',
        ],
    },
    {
        'id':       'compounding_overgen_af',
        'label':    'COMPOUNDING OVERGENERALIZATION - AF anticoagulation cascade',
        'source':   'prewritten',
        'question': 'Should patients with atrial fibrillation take anticoagulants?',
        'expect':   'all_signals',
        'steps': [
            'Atrial fibrillation causes disorganized atrial electrical activity and loss of effective atrial contraction.',
            'Blood stasis in the left atrial appendage during AF promotes local thrombus formation.',
            'Thrombi originating in the left atrial appendage can embolize to cerebral arteries, causing ischemic stroke.',
            'Direct oral anticoagulants such as apixaban and rivaroxaban reduce stroke risk in AF patients by approximately 65%.',
            'Since anticoagulation reduces thromboembolic events, all patients with irregular heart rhythms should receive it.',
            'Any patient diagnosed with a cardiac arrhythmia is at elevated thromboembolic risk and warrants anticoagulation therapy.',
            'Anticoagulation is therefore broadly indicated for patients with cardiac disease to prevent adverse vascular events.',
        ],
    },
]

ALL_CHAINS = list(PREWRITTEN_COTS)
BATCH_RESULTS = []

print(f'{len(ALL_CHAINS)} chains loaded:\n')
for c in ALL_CHAINS:
    print(f'  [{c["expect"]:20s}]  {c["id"]}')

In [ ]:
# ── 5. Stage 2: Concept extraction ───────────────────────────────────────────
def _strip_md(text):
    text = _re.sub(r'^#+\s*', '', text, flags=_re.MULTILINE)
    text = _re.sub(r'\*\*(.+?)\*\*', r'\1', text)
    text = _re.sub(r'\*(.+?)\*',     r'\1', text)
    text = _re.sub(r'`(.+?)`',       r'\1', text)
    return text.strip()

print('Stage 2 — Concept extraction...')
t0 = time.time()
for chain in ALL_CHAINS:
    chain['steps_clean'] = [_strip_md(s) for s in chain['steps']]
    chain['concepts']    = extract_concepts(chain['steps_clean'],
                                            scispacy_when='never', top_k=5)
    n_concepts = sum(len(step) for step in chain['concepts'])
    print(f'  {chain["id"]:42s}  {n_concepts} concepts')
print(f'Done in {time.time()-t0:.1f}s')

In [ ]:
# ── 6. Stage 3: NLI entailment ────────────────────────────────────────────────
print('Stage 3 — NLI entailment...')
t0 = time.time()
for chain in ALL_CHAINS:
    chain['pairs'] = build_entailment_records(
        chain['steps_clean'], chain['concepts']
    )
    n_contra = sum(1 for p in chain['pairs'] if p['final_label'] == 'contradiction')
    print(f'  {chain["id"]:42s}  {len(chain["pairs"])} pairs  {n_contra} contradiction(s)')
print(f'Done in {time.time()-t0:.1f}s')

In [ ]:
# ── 7. Stage 5: UMLS signals ─────────────────────────────────────────────────
print('Stage 5 — UMLS signals...')
t0 = time.time()
for chain in ALL_CHAINS:
    chain['density']     = score_density(chain['concepts'], chain['steps_clean'])
    chain['specificity'] = score_specificity(chain['concepts'])
    den_sl  = chain['density'].get('slope')
    spec_sl = chain['specificity'].get('depth_slope')
    den_str  = f'{den_sl:+.4f}'  if den_sl  is not None else '   n/a '
    spec_str = f'{spec_sl:+.4f}' if spec_sl is not None else '   n/a '
    print(f'  {chain["id"]:42s}  den_slope={den_str}  spec_slope={spec_str}')
print(f'Done in {time.time()-t0:.1f}s')

In [ ]:
# ── 8. Results summary ────────────────────────────────────────────────────────
rows = []
for chain in ALL_CHAINS:
    den  = chain.get('density',     {})
    spec = chain.get('specificity', {})
    rows.append({
        'ID':         chain['id'],
        'Expect':     chain['expect'],
        'Den slope':  round(den.get('slope',  float('nan')), 4),
        'Den risk':   round(den.get('overall_risk', float('nan')), 3),
        'Spec slope': round(spec.get('depth_slope', float('nan')), 4),
        'Spec risk':  round(spec.get('overall_specificity_score', float('nan')), 3),
        'Leaps':      len(spec.get('abstraction_leaps', [])),
        'Contra':     sum(1 for p in chain.get('pairs', [])
                         if p['final_label'] == 'contradiction'),
    })

df = pd.DataFrame(rows)
df.style.background_gradient(subset=['Den risk', 'Spec risk'], cmap='RdYlGn_r')

In [ ]:
# ── 9. Stage 6a: NLI P(contradiction) matrix, all chains ─────────────────────
#
# N×N matrix where cell [i,j] = P(contradiction) for pair i->j.
# Only adjacent pairs are scored; non-adjacent cells are grey (NaN).

ncols = 2
nrows = math.ceil(len(ALL_CHAINS) / ncols)
fig, axes = plt.subplots(nrows, ncols, figsize=(14, nrows * 3.5),
                         squeeze=False, constrained_layout=True)

for ci, chain in enumerate(ALL_CHAINS):
    r, c    = divmod(ci, ncols)
    ax      = axes[r, c]

    steps_c = chain.get('steps_clean', chain['steps'])
    pairs   = chain.get('pairs', [])
    n_steps = len(steps_c)

    mat = np.full((n_steps, n_steps), np.nan)
    for p in pairs:
        pi, pj = p['step_pair']
        if pi < n_steps and pj < n_steps:
            mat[pi, pj] = p['probs'].get('contradiction', 0)

    ax.imshow(mat, vmin=0, vmax=1, cmap='RdYlGn_r', aspect='auto')
    ax.set_xticks(range(n_steps))
    ax.set_yticks(range(n_steps))
    ax.set_xticklabels([f'S{i}' for i in range(n_steps)],
                       rotation=45, ha='right', fontsize=7)
    ax.set_yticklabels([f'S{i}' for i in range(n_steps)], fontsize=7)
    ax.set_xlabel('j (hyp)', fontsize=7)
    ax.set_ylabel('i (prem)', fontsize=7)

    for p in pairs:
        pi, pj = p['step_pair']
        if pi < n_steps and pj < n_steps:
            val = mat[pi, pj]
            lbl = p['final_label'][0].upper()
            ax.text(pj, pi, f'{lbl}\n{val:.2f}',
                    ha='center', va='center', fontsize=7,
                    color='white' if val > 0.6 else 'black')

    ax.set_title(chain['label'][:38], fontsize=8, pad=3)

if len(ALL_CHAINS) % 2:
    axes[nrows - 1, 1].set_visible(False)

fig.suptitle(
    '6a - NLI P(contradiction) per Step-Pair  (E=entailment  N=neutral  C=contradiction | colour 0-1)',
    fontsize=10, y=1.01)
plt.show()

In [ ]:
# ── 10. Stage 6b: UMLS concept density per step, all chains ──────────────────
#
# Per-step: density = valid_concept_count / word_count.
# Negative slope = concept grounding decreasing (leakage signal).

ncols = 2
nrows = math.ceil(len(ALL_CHAINS) / ncols)
fig, axes = plt.subplots(nrows, ncols, figsize=(12, nrows * 3),
                         squeeze=False, constrained_layout=True)

for ci, chain in enumerate(ALL_CHAINS):
    r, c    = divmod(ci, ncols)
    ax      = axes[r, c]
    density = chain.get('density', {})

    if not density.get('configured'):
        ax.text(0.5, 0.5, 'UMLS not configured', transform=ax.transAxes,
                ha='center', va='center', fontsize=9, color='grey', style='italic')
        ax.set_title(chain['label'][:38], fontsize=8, pad=3)
        continue

    per_step = density.get('per_step', [])
    xs = [row['step_index'] for row in per_step]
    ys = [row['density']    for row in per_step]

    ax.bar(xs, ys, color='#4C72B0', alpha=0.6, width=0.6, label='Density')
    ax.plot(xs, ys, 'o', color='#4C72B0', markersize=4)

    if len(xs) >= 2:
        coeffs   = np.polyfit(xs, ys, 1)
        trend_ys = np.polyval(coeffs, xs)
        ax.plot(xs, trend_ys, '--', color='#C44E52', linewidth=1.5, label='Trend')

    slope = density.get('slope')
    if slope is not None:
        ax.text(0.98, 0.97, f'slope={slope:.4f}', transform=ax.transAxes,
                ha='right', va='top', fontsize=7,
                color='#C44E52' if slope < 0 else '#2ca02c')

    onset = density.get('leakage_onset_step')
    if onset is not None:
        ax.axvline(onset, color='red', linestyle=':', linewidth=1.2, alpha=0.7,
                   label=f'Onset S{onset}')

    ax.set_xticks(xs)
    ax.set_xticklabels([f'S{x}' for x in xs], fontsize=7)
    ax.set_ylabel('Concepts / word', fontsize=7)
    ax.set_title(chain['label'][:38], fontsize=8, pad=3)
    ax.legend(fontsize=6, loc='upper left', framealpha=0.6)
    ax.grid(axis='y', alpha=0.3)

if len(ALL_CHAINS) % 2:
    axes[nrows - 1, 1].set_visible(False)

fig.suptitle('6b - UMLS Concept Density per Step  (concepts/word, negative slope = grounding loss)',
             fontsize=10, y=1.01)
plt.show()

In [ ]:
# ── 11. Stage 6c: UMLS ontology depth per step, all chains ───────────────────
#
# Per-step: avg_depth = average ancestor count for grounded concepts.
# Higher depth = more specific; negative slope = abstracting (leakage).

ncols = 2
nrows = math.ceil(len(ALL_CHAINS) / ncols)
fig, axes = plt.subplots(nrows, ncols, figsize=(12, nrows * 3),
                         squeeze=False, constrained_layout=True)

for ci, chain in enumerate(ALL_CHAINS):
    r, c        = divmod(ci, ncols)
    ax          = axes[r, c]
    specificity = chain.get('specificity', {})

    if not specificity.get('configured'):
        ax.text(0.5, 0.5, 'UMLS not configured', transform=ax.transAxes,
                ha='center', va='center', fontsize=9, color='grey', style='italic')
        ax.set_title(chain['label'][:38], fontsize=8, pad=3)
        continue

    per_step = specificity.get('per_step', [])
    xs_all   = [row['step_index'] for row in per_step]
    ys_all   = [row['avg_depth']  for row in per_step]

    valid = [(x, y) for x, y in zip(xs_all, ys_all) if y is not None]
    if valid:
        vxs, vys = zip(*valid)
        ax.plot(vxs, vys, '^-', color='#55A868', linewidth=1.8,
                markersize=5, label='Depth')
        if len(vxs) >= 2:
            coeffs   = np.polyfit(vxs, vys, 1)
            trend_ys = np.polyval(coeffs, vxs)
            ax.plot(vxs, trend_ys, '--', color='#C44E52', linewidth=1.5, label='Trend')

    slope = specificity.get('depth_slope')
    if slope is not None:
        ax.text(0.98, 0.97, f'slope={slope:.4f}', transform=ax.transAxes,
                ha='right', va='top', fontsize=7,
                color='#C44E52' if slope < 0 else '#2ca02c')

    for leap in specificity.get('abstraction_leaps', []):
        ax.axvspan(leap['from_step'] - 0.3, leap['to_step'] + 0.3,
                   alpha=0.15, color='orange',
                   label=f'Leap {leap["from_step"]}->{leap["to_step"]}')

    if xs_all:
        ax.set_xticks(xs_all)
        ax.set_xticklabels([f'S{x}' for x in xs_all], fontsize=7)
    ax.set_ylabel('Ontology depth (ancestors)', fontsize=7)
    ax.set_title(chain['label'][:38], fontsize=8, pad=3)
    ax.legend(fontsize=6, loc='upper left', framealpha=0.6)
    ax.grid(axis='y', alpha=0.3)

if len(ALL_CHAINS) % 2:
    axes[nrows - 1, 1].set_visible(False)

fig.suptitle('6c - UMLS Ontology Depth per Step  (ancestors, negative slope = abstracting, orange = leap)',
             fontsize=10, y=1.01)
plt.show()